# PrivAware v2 — Day 4: Privacy Policy Classifier

**Goal:** Train a second NLP model that reads privacy policy text and classifies it as Safe, Risky, or Deceptive. Then update the live API to use both models.

**Before you start:**
- Runtime → Change runtime type → **T4 GPU** → Save
- Run cells one by one (Shift+Enter)

**What you get today:**
- Trained privacy policy classifier on HuggingFace Hub
- Updated FastAPI with both /scan-url AND /scan-policy endpoints live

## Cell 1 — Install libraries

In [ ]:
!pip install transformers datasets torch scikit-learn huggingface_hub -q
print('Libraries installed!')

## Cell 2 — Login to HuggingFace

In [ ]:
from huggingface_hub import notebook_login
print('Paste your HuggingFace WRITE token below.')
print('Get it from: huggingface.co -> Settings -> Access Tokens')
notebook_login()

## Cell 3 — Load ToS;DR dataset
ToS;DR (Terms of Service; Didn't Read) is a dataset of privacy policy clauses labeled as good, bad, or neutral. Perfect for our use case.

In [ ]:
from datasets import load_dataset
import pandas as pd

print('Loading ToS;DR dataset...')
dataset = load_dataset('toloka-ai/ToSDR')

df = pd.DataFrame(dataset['train'])
print(f'Loaded {len(df)} rows')
print(f'Columns: {df.columns.tolist()}')
print('\nSample rows:')
print(df.head())
print('\nLabel distribution:')
print(df['label'].value_counts())

## Cell 4 — Clean and map labels

In [ ]:
print('Original labels:', df['label'].unique())

# Map to 3 classes: 0=Safe, 1=Risky, 2=Deceptive
label_map = {
    'good': 0,
    'neutral': 1,
    'bad': 2,
    # handle any alternate naming
    0: 0, 1: 1, 2: 2
}

# Check what text column is called
text_col = 'text' if 'text' in df.columns else df.columns[0]
print(f'Text column: {text_col}')

df = df.rename(columns={text_col: 'text'})
df['label'] = df['label'].map(label_map)
df = df.dropna(subset=['text', 'label'])
df = df[df['text'].str.len() > 10]
df['label'] = df['label'].astype(int)

print(f'\nAfter cleaning: {len(df)} rows')
print('Label distribution:')
label_names = {0: 'Safe', 1: 'Risky', 2: 'Deceptive'}
for k, v in df['label'].value_counts().items():
    print(f'  {label_names[k]} ({k}): {v}')

## Cell 5 — Split into train/val/test

In [ ]:
from sklearn.model_selection import train_test_split

X = df['text'].tolist()
y = df['label'].tolist()

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_val, X_test, y_val, y_test     = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

## Cell 6 — Tokenize

In [ ]:
from transformers import DistilBertTokenizerFast
import torch

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(texts, labels, max_length=256):
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_length, return_tensors='pt')
    return encodings, torch.tensor(labels)

print('Tokenizing...')
train_enc, train_lab = tokenize(X_train, y_train)
val_enc,   val_lab   = tokenize(X_val,   y_val)
test_enc,  test_lab  = tokenize(X_test,  y_test)
print('Done!')

## Cell 7 — Create datasets and loaders

In [ ]:
from torch.utils.data import Dataset, DataLoader

class PolicyDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_loader = DataLoader(PolicyDataset(train_enc, train_lab), batch_size=16, shuffle=True)
val_loader   = DataLoader(PolicyDataset(val_enc,   val_lab),   batch_size=16)
test_loader  = DataLoader(PolicyDataset(test_enc,  test_lab),  batch_size=16)

print(f'Train batches: {len(train_loader)}')
print('Datasets ready!')

## Cell 8 — Load model and train
3-class classification this time (Safe / Risky / Deceptive). Takes ~20-30 minutes.

In [ ]:
from transformers import DistilBertForSequenceClassification, get_scheduler
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
NUM_EPOCHS = 3
scheduler = get_scheduler('linear', optimizer=optimizer, num_warmup_steps=50, num_training_steps=NUM_EPOCHS*len(train_loader))

def evaluate(loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labs = batch['labels'].to(device)
            out  = model(input_ids=ids, attention_mask=mask)
            preds = torch.argmax(out.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labs.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='weighted')
    return acc, f1

best_val_acc = 0
history = []
print('Training started...')
print('='*55)

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    start = time.time()
    for step, batch in enumerate(train_loader):
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labs = batch['labels'].to(device)
        out  = model(input_ids=ids, attention_mask=mask, labels=labs)
        out.loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += out.loss.item()
        if step % 50 == 0:
            print(f'  Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {out.loss.item():.4f}')
    val_acc, val_f1 = evaluate(val_loader)
    history.append({'epoch': epoch+1, 'loss': total_loss/len(train_loader), 'val_acc': val_acc, 'val_f1': val_f1})
    print(f'\nEpoch {epoch+1}: Loss={total_loss/len(train_loader):.4f} | Val Acc={val_acc*100:.2f}% | Val F1={val_f1:.4f} | Time={time.time()-start:.0f}s')
    print('='*55)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained('/content/policy_model')
        tokenizer.save_pretrained('/content/policy_model')
        print('  *** Best model saved ***')

print('\nTraining complete!')

## Cell 9 — Test results

In [ ]:
from sklearn.metrics import classification_report
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained('/content/policy_model')
model.to(device)

test_acc, test_f1 = evaluate(test_loader)

print('='*55)
print('FINAL RESULTS — NOTE THESE FOR YOUR PANEL')
print('='*55)
print(f'Test Accuracy : {test_acc*100:.2f}%')
print(f'Test F1 Score : {test_f1:.4f}')
print('='*55)

## Cell 10 — Push policy model to HuggingFace Hub

In [ ]:
YOUR_ACCURACY = '00.00'  # paste your test accuracy here
YOUR_F1       = '0.00'   # paste your F1 score here
REPO_ID = 'V3d4nt7/privaware-policy-classifier'

model.config.id2label = {0: 'SAFE', 1: 'RISKY', 2: 'DECEPTIVE'}
model.config.label2id = {'SAFE': 0, 'RISKY': 1, 'DECEPTIVE': 2}

model_card = f"""---
language: en
tags:
- text-classification
- privacy-policy
- cybersecurity
- distilbert
datasets:
- toloka-ai/ToSDR
---

# PrivAware Privacy Policy Classifier

DistilBERT fine-tuned to classify privacy policy clauses as Safe, Risky, or Deceptive.

## Details
- Base model: distilbert-base-uncased
- Dataset: toloka-ai/ToSDR
- Test Accuracy: {YOUR_ACCURACY}%
- Test F1: {YOUR_F1}

## Labels
- SAFE (0)
- RISKY (1)
- DECEPTIVE (2)
"""

with open('/content/policy_model/README.md', 'w') as f:
    f.write(model_card)

print(f'Pushing to: https://huggingface.co/{REPO_ID}')
model.push_to_hub(REPO_ID)
tokenizer.push_to_hub(REPO_ID)
print('Policy model pushed!')

## Cell 11 — Update the FastAPI backend with both models
This rewrites app.py to use both the phishing model AND the policy model, then redeploys the Space.

In [ ]:
from huggingface_hub import HfApi

app_code = '''
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from transformers import pipeline
import uvicorn

app = FastAPI(title="PrivAware API", version="2.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

print("Loading phishing model...")
phishing_classifier = pipeline(
    "text-classification",
    model="V3d4nt7/privaware-phishing-detector"
)

print("Loading policy model...")
policy_classifier = pipeline(
    "text-classification",
    model="V3d4nt7/privaware-policy-classifier"
)

print("Both models loaded!")

class URLRequest(BaseModel):
    url: str

class PolicyRequest(BaseModel):
    text: str

@app.get("/")
def root():
    return {"status": "ok", "message": "PrivAware API v2 running — both models live"}

@app.get("/health")
def health():
    return {"status": "healthy", "models": ["phishing-detector", "policy-classifier"]}

@app.post("/scan-url")
def scan_url(request: URLRequest):
    if not request.url:
        raise HTTPException(status_code=400, detail="URL is required")
    try:
        result = phishing_classifier(request.url[:512])[0]
        label = result["label"]
        confidence = round(result["score"] * 100, 2)
        risk_score = round(confidence) if label == "PHISHING" else round(100 - confidence)
        return {
            "url": request.url,
            "label": label,
            "confidence": confidence,
            "risk_score": risk_score,
            "is_phishing": label == "PHISHING"
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/scan-policy")
def scan_policy(request: PolicyRequest):
    if not request.text:
        raise HTTPException(status_code=400, detail="Text is required")
    try:
        result = policy_classifier(request.text[:512])[0]
        label = result["label"]
        confidence = round(result["score"] * 100, 2)
        risk_map = {"SAFE": 10, "RISKY": 55, "DECEPTIVE": 90}
        risk_score = risk_map.get(label, 50)
        return {
            "label": label,
            "confidence": confidence,
            "risk_score": risk_score,
            "is_deceptive": label == "DECEPTIVE"
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/scan-full")
def scan_full(request: URLRequest):
    url_result = scan_url(request)
    combined_score = url_result["risk_score"]
    return {
        "url": request.url,
        "phishing": url_result,
        "combined_risk_score": combined_score
    }

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=7860)
'''

with open('/content/app.py', 'w') as f:
    f.write(app_code)

api = HfApi()
api.upload_file(
    path_or_fileobj='/content/app.py',
    path_in_repo='app.py',
    repo_id='V3d4nt7/privaware-api',
    repo_type='space'
)

print('Updated app.py uploaded to Space!')
print('Space is rebuilding with both models...')
print('Watch: https://huggingface.co/spaces/V3d4nt7/privaware-api')
print('Wait for RUNNING status then run Cell 12.')

## Cell 12 — Final test: both endpoints working

In [ ]:
import requests

SPACE_URL = 'https://V3d4nt7-privaware-api.hf.space'

print('Testing /scan-url...')
r = requests.post(f'{SPACE_URL}/scan-url', json={'url': 'http://paypal-secure-login.xyz/verify'})
print(r.json())

print('\nTesting /scan-policy...')
r = requests.post(f'{SPACE_URL}/scan-policy', json={'text': 'We share your personal data with third party advertisers without your explicit consent.'})
print(r.json())

print('\n' + '='*55)
print('DAY 4 COMPLETE!')
print('Both models live. API has 3 endpoints:')
print(f'  GET  {SPACE_URL}/health')
print(f'  POST {SPACE_URL}/scan-url')
print(f'  POST {SPACE_URL}/scan-policy')
print('Message me for Day 5 (Chrome extension).')
print('='*55)